In [1]:
from device.Relay import KmtronicRelayCh4
from system.fluidics_exchange_system import *
import os
import json

In [4]:
with open(os.path.join("config_file", "KMtronic_relay.json"), 'r') as r:
    Relay_cfg = json.load(r)

FileNotFoundError: [Errno 2] No such file or directory: 'config_file\\KMtronic_relay.json'

In [5]:
relay=KmtronicRelayCh4(Relay_cfg[0]['port'])
relay.on_connect_requested()


[False, False, False, False]


In [6]:
#by pass
relay.change_relay_state(0, True)

[True, False, False, False]


In [6]:
# go charm
relay.change_relay_state(0, False)

[False, False, False, False]


In [9]:
relay.quit()

DEBUG... cleaning up Relay


In [15]:
## Test Relay from port directly

In [12]:
import serial
import time
_serial_con = serial.Serial(
                "COM4",
                9600,
                8,
                serial.PARITY_NONE,
                1,
                timeout=1.0)

In [13]:
_serial_con.is_open

True

In [17]:
RESPONSE_TIME = 0.2
def _send_cmd(cmd: bytearray):
    if _serial_con and _serial_con.is_open:
        _serial_con.reset_input_buffer()
        _serial_con.reset_output_buffer()
        _serial_con.write(cmd)
        _serial_con.flush()
def _get_response():
    val = _serial_con.readline()
    return val

In [19]:
_send_cmd(bytearray([255, 9, 0]))
time.sleep(3)
_lks = [bool(x) for x in _get_response()]


In [20]:
_lks

[False, False, False, False]

In [21]:
def _get_relay_states(_lks):
    """
    Returns list of relay states as true or false vals depending on each
        state.
    Example: [True, False, False, True] would indicate that the first and
        fourth relay are currently on.
    :returns: List of bools
    """
    _send_cmd(bytearray([255, 9, 0]))
    time.sleep(RESPONSE_TIME)
    _lks = [bool(x) for x in _get_response()]
    return _lks

In [22]:
def operate_relay(_lks,relay_number: int, state: bool = False, verify: bool = False):
    if _lks[relay_number] != state:
        _send_cmd(bytearray([255, relay_number + 1, state]))
        if verify:
           time.sleep(RESPONSE_TIME)
           _lks = _get_relay_states()  # might want to just check the device
        else:
           _lks[relay_number] = state

In [24]:
operate_relay(_lks,0, True)

In [25]:
_serial_con.close()

In [2]:
pos_path="D:\\"
fluidics=FluidicSystem(pos_path)

In [4]:
fluidics.config_relay()

[False, False, False, False]
DEBUG... cleaning up Relay
disconnect relay,close the port!
disconeect relay, close the port!


In [5]:
fluidics.connect_relay()

disconnect relay,close the port!
[False, False, False, False]
[True, False, False, False]


In [7]:
SELECT_BYPASS_STATE = True
SELECT_CHAMBER_STATE = not SELECT_BYPASS_STATE

In [14]:
fluidics.select_bypass(SELECT_BYPASS_STATE)

[True, False, False, False]
[True, True, False, False]
[True, True, True, False]


In [15]:
fluidics.select_chamber(SELECT_CHAMBER_STATE)

[False, True, True, False]
[False, False, True, False]
[False, False, False, False]
